# MCMC using `emcee`

In [ ]:
import numpy as np
import scipy as sp 
import emcee
# plotting package for visualizing multidimensional sampling
import corner
import matplotlib.pyplot as plt
%matplotlib inline
np.random.seed(1)

Let's return to our bimodal distribution from earlier

In [ ]:
_p0 = sp.stats.norm(loc=1, scale=6)
_p1 = sp.stats.norm(loc=-2, scale=1)
def logp(x): # emcee takes in logp as input
    return np.log(0.5*(_p0.pdf(x) + _p1.pdf(x)))

This time we'll use `emcee` to sample it

In [ ]:
ndim = 1 # dimension of p(x)
nwalkers = 32
nsteps = 1000

In [ ]:
# initialize the walkers
pos = np.random.uniform(-0.1, 0.1, size=(nwalkers, ndim))

In [ ]:
# run the MCMC Ensemble Sampler
sampler = emcee.EnsembleSampler(nwalkers, ndim, logp)
_ = sampler.run_mcmc(pos, nsteps, progress=True)

Let's first take a look at the walkers

In [ ]:
samples = sampler.get_chain(flat=False)

plt.figure(figsize=(8,2))
for i in range(nwalkers): 
    plt.plot(samples[:,i,0], 'k', alpha=0.1, rasterized=True)
plt.xlim(0, 200)
plt.ylabel('x');plt.xlabel('step number');plt.show()

You can see **burn-in**, where the chain reflects the initialization and not $p(x)$, in the very beginning at steps 0 to ~50.

Let's discard the burnin, conservatively let's say 200 steps, and take a look at the sampling.

In [ ]:
burnin = 200
samples = sampler.get_chain(discard=burnin, flat=True)

In [ ]:
plt.figure(figsize=(4,3))
plt.hist(samples, range=(-20, 20), bins=40, density=True)
plt.plot(np.linspace(-20, 20, 100), np.exp(logp(np.linspace(-20, 20, 100))), c='k', ls='--')
plt.xlabel('$x$', fontsize=20)
plt.ylabel('$p(x)$', fontsize=20); plt.show()

# `emcee` for Bayesian inference
Now let's put the pieces together for an iconic Bayesian inference problem of fitting a model to data

In [ ]:
a_true, b_true = 0, 1.75
x = np.array([1,1.5,2,2.5])
y = np.array([1.87640523, 2.59001572, 3.4978738 , 4.47408932])
yerr = 0.1

In [ ]:
plt.figure(figsize=(4,3))
plt.errorbar(x, y, yerr=yerr, fmt='.k')
plt.xlabel('x'); plt.ylabel('y')

We want to fit a simple linear model: 
$m(x) = a + b x$, where $\theta = \{a, b\}$ are the free parameters

In [ ]:
def model(theta, x):
    a, b = theta
    return a + b * x

In [ ]:
plt.figure(figsize=(4,3))
plt.errorbar(x, y, yerr=yerr, fmt='.k')
plt.plot(np.linspace(1, 2.5, 10), model([1, 1], np.linspace(1, 2.5, 10)), c='b')
plt.xlabel('x'); plt.ylabel('y')

We can write down the likelihood, prior, and posterior as follows: 

In [ ]:
def log_likelihood(theta, x, y, yerr):
    m = model(theta, x)
    return -0.5 * np.sum((y - m) ** 2 / yerr**2)

def log_prior(theta):
    a, b = theta
    if -10. < a < 10. and -10.0 < b < 10.0:
        return 0.0
    return -np.inf

def log_posterior(theta, x, y, yerr):
    lp = log_prior(theta)
    if not np.isfinite(lp): return -np.inf
    return lp + log_likelihood(theta, x, y, yerr)

Now to sample the posterior using `emcee`

In [ ]:
ndim = 2 # p(theta | X) is 2D
nwalkers = 32 

we need to initialize the walkers. It's common to initialize the walkers near the MLE or MAP for reduce burn-in and faster convergence$^{*}$

$^*$ this intuition is for a unimodal and well-behaved posterior

In [ ]:
# start at the MLE
theta_guess = [1., 1.] # initial guess 
def neg_log_likelihood(theta, x, y, yerr): 
    return -1 * log_likelihood(theta, x, y, yerr)
theta_est = sp.optimize.fmin(neg_log_likelihood, 
                             theta_guess, 
                             args=(x, y, yerr))
print('theta_MLE =', theta_est)

# create a small ball around the MLE the initialize each walker 
pos = theta_est + 1e-4 * np.random.randn(nwalkers, ndim)

Now we can run the `emcee` EnsembleSampler. Let's start with $N=200$

In [ ]:
# run emcee
sampler = emcee.EnsembleSampler(nwalkers, ndim, 
                                log_posterior, 
                                args=(x, y, yerr))
_ = sampler.run_mcmc(pos, 200, progress=True)

Let's examine the walkers to check burn-in and convergence

In [ ]:
fig, axes = plt.subplots(ndim, sharex=True, figsize=(8,2))
samples = sampler.get_chain()
labels = ["a", "b"]
for i in range(ndim):
    ax = axes[i]
    ax.plot(samples[:, :, i], "k", alpha=0.3, rasterized=True)
    ax.set_xlim(0, 200); ax.set_ylabel(labels[i])
axes[-1].set_xlabel("step number")

You can clearly see the burn-in. Let's discard the first 100 steps to be safe.

In [ ]:
burnin = 100 

Lets check the posterior $p(\theta|X)$ against $\theta_{\rm true} = [0, 1.75]$

In [ ]:
flat_samples = sampler.get_chain(discard=100, flat=True, thin=15)

fig = plt.figure(figsize=(4,4))
fig = corner.corner(flat_samples, labels=labels, 
                    truths=[a_true, b_true], fig=fig)

`emcee` also has a build in **autocorrelation time** estimator

In [ ]:
tau = sampler.get_autocorr_time(discard=burnin)
print(tau)

You should've run into an error above because we only ran $N=200$.

1. Run the chain for a range of iteratinos $N = 2000 - 10000$ and calculate the autocorrelation time. 

In [ ]:
N_iters, taus = [], []
for N in : # loop over different number of iteractions
    # run emcee for N steps

    # calculate autocorrelation time
    _tau = sampler.get_autocorr_time(discard=burnin)

    N_iters.append(N)
    taus.append(_tau)

In [ ]:
fig = plt.figure(figsize=(8,3))
sub = fig.add_subplot(121)
sub.plot(N_iters, np.array(taus)[:,0])
sub.set_xlabel("$N$", fontsize=20)
sub.set_xscale('log')
sub.set_ylabel(r"$\tau_a$", fontsize=20)
sub = fig.add_subplot(122)
sub.plot(N_iters, np.array(taus)[:,1])
sub.set_xlabel("$N$", fontsize=20)
sub.set_xscale('log')
sub.set_ylabel(r"$\tau_b$", fontsize=20)

2. Based on how many iterations do you need for convergence? 

*hint: $N > 50\tau$ and $\tau$ converges to percent level.*

3. Discard burn-in, thin the chain, and make a corner plot of $(a,b)$ with the converged MCMC chain. Overplot `a_true` and `b_true`, $\theta_{\rm true} = [0, 1.75]$